In [ ]:
####################################################################################################
# Complete Port Scanner with Banner Grabbing, Nmap Scan, Shodan Lookup, CSV + HTML Report
# -----------------------------------------------------------------------------------
# This tool performs layered network reconnaissance against a target host.
# -----------------------------------------------------------------------------------
# Features:                                                                                       #
# - Accepts target host and port range                                                            #
# - Scans ports using socket                                                                      #
# - Banner grabbing from open ports                                                               #
# - Shodan enrichment (optional)                                                                  #
# - Nmap scan (-sV) for detailed service info       (optional)                                    #                                                                    #
# - Multithreading for faster scanning                                                            #
# - Progress bar with tqdm                                                                        #
# - Detects common services from banner keywords  
# - Optional Nmap scan
# - CSV output with timestamp 
# - HTML report
####################################################################################################
####################################################################################################


In [6]:
# Import libraries required


import socket
from datetime import datetime
import csv
import argparse
from concurrent.futures import ThreadPoolExecutor, as_completed
import sys

# Optional libraries
try:
    import shodan
except ImportError:
    shodan = None

try:
    import nmap
except ImportError:
    nmap = None

try:
    from tqdm import tqdm
except ImportError:
    tqdm = None


In [7]:
# -----------------------------------------------------------------------------------
# SECTION 1 — Resolve Hostname to IP address
# -----------------------------------------------------------------------------------
# Before performing any network scan, the hostname must be translated
# into an IPv4 address using DNS resolution.
#
# This ensures the scanner communicates directly with the target host
# using its IP address instead of repeatedly performing DNS lookups.
# -----------------------------------------------------------------------------------

def resolve_host(hostname):

    try:
        ip = socket.gethostbyname(hostname)     # This performs DNS lookup and returns the IPv4 address
        return ip
    except socket.error:
        print(f"[!] Could not resolve host: {hostname}")
        return None

In [8]:
# -----------------------------------------------------------------------------------
# SECTION 2 — Socket Port Scanner + Banner Grabbing
# -----------------------------------------------------------------------------------
# This function performs a TCP connect scan on a single port.
# This works by attempting to establish a full TCp connection to the target port
# If the connection succeeds, the port is considered open.
# After connecting, the scanner attempts to read any banner data sent by the service, 
# which may reveal application information
# -----------------------------------------------------------------------------------



def scan_port(host, port):
    #Scan a single TCP port

    tcp_socket = socket.socket(socket.AF_INET, socket.SOCK_STREAM)    # Create a TCP socket using IPv4 (AF_INET) and TCP protocol (SOCK_STREAM)  
    tcp_socket.settimeout(1)                                          # Set a timeout to avoid long delays on filtered or unresponsive ports

    banner = ""
    is_open = False

    try:
        result = tcp_socket.connect_ex((host, port))

        # If connection succeeds
        if result == 0:                                                     # 0 means connection successful (port open)
            is_open = True


            try:                                                            # Try grabbing banner after a TCP connection is established
                tcp_socket.settimeout(0.5)
                data = tcp_socket.recv(1024)                                # Read up to 1024 bytes from the service
                banner = data.decode("utf-8", errors="ignore").strip()      # Decode banner safely (ignore non-UTF8 bytes)
            except (socket.timeout, socket.error):
                banner = ""
            except Exception:
                banner = ""
    except (socket.timeout, socket.error):
        is_open = False
    except Exception as e:
        print(f"[!] Error scanning port {port}: {e}")
        is_open = False

    # Always close the socket to release system resources
    finally:
        tcp_socket.close()

    return is_open, banner

In [9]:
# -----------------------------------------------------------------------------------
# SECTION 3 — Nmap Scan
# -----------------------------------------------------------------------------------
# Nmap provides deeper service fingerprinting than simple socket scanning.
#
# The -sV option performs version detection by sending specialized probes
# to identify the exact service and application version running on a port.
#
# This provides far richer intelligence than banner grabbing alone.
# -----------------------------------------------------------------------------------


def nmap_scan(host, ports):
    #Run Nmap scan(-sV) for detailed service detection

    results = []

    if not nmap:
        print("[!] python-nmap not installed")
        return results

    try:
        print("\n[+] Running Nmap service detection scan (-sV)...")

        nm = nmap.PortScanner()
        nm.scan(host, ports, arguments="-sV")                                # Execute the Nmap scan

        for h in nm.all_hosts():                                             # Parse results returned by Nmap
            for proto in nm[h].all_protocols():
                for port in nm[h][proto]:

                    info = nm[h][proto][port]

                    results.append({
                        "port": port,
                        "state": info["state"],
                        "service": info["name"],
                        "version": info.get('version', ''),
                        "banner": info.get("product", "")
                    })

        print("[+] Nmap scan completed.\n")

    except Exception as e:                                                  # Handle errors
        print(f"[!] Nmap scan failed: {e}")

    return results


In [10]:
# -----------------------------------------------------------------------------------
# SECTION 4 — Shodan Lookup
# -----------------------------------------------------------------------------------
# Function to query shodan throuh its API
# Querying Shodan allows the scanner to enrich results with OSINT data
# such as organization, operating system, country, and previously
# discovered service banners.
# -----------------------------------------------------------------------------------


def shodan_lookup(ip, api_key):

    info = {
        "org": "",
        "os": "",
        "country": "",
        "ports": [],
        "banners": []
    }

    if not shodan:
        print("[!] Shodan library not installed")
        return info

    if not api_key:
        return info

    try:
        api = shodan.Shodan(api_key)
        result = api.host(ip)                                       # Query Shodan database for this host

        info["org"] = result.get("org", "")
        info["os"] = result.get("os", "")
        info["country"] = result.get("country_name", "")
        info["ports"] = result.get("ports", [])
        for item in result.get('data', []):
            banner = item.get('data', '').strip()
            if banner:
                info['banners'].append(banner)

    except shodan.APIError as e:
        print(f"[!] Shodan API error: {e}")

    return info


In [11]:
# -----------------------------------------------------------------------------------
# SECTION 5 — Save CSV Report
# -----------------------------------------------------------------------------------
# Function to save results to CSV
# CSV output allows scan results to be easily analysed using spreadsheets,
# SIEM tools, or additional scripts.
# -----------------------------------------------------------------------------------


def save_to_csv(results, filename="scan_results.csv"):

    fields = ["timestamp", "host", "ip", "port", "state", "service", "banner", "notes"]

    with open(filename, "w", newline="", encoding="utf-8") as f:

        writer = csv.DictWriter(f, fieldnames=fields)

        writer.writeheader()

        for row in results:
            writer.writerow(row)

    print(f"\n[+] CSV saved: {filename}")

In [12]:
# -----------------------------------------------------------------------------------
# SECTION 6 — Save HTML Report
# -----------------------------------------------------------------------------------
# HTML output provides a human-readable report that can be viewed
# directly in a web browser.
# -----------------------------------------------------------------------------------

def save_to_html(results, filename="scan_report.html"):

    with open(filename, "w", encoding="utf-8") as f:

        f.write("<html>")
        f.write("<head><title>Port Scan Report</title></head>")
        f.write("<body>")

        f.write("<h2>Port Scan Report</h2>")
        f.write("<table border='1' cellpadding='5'>")

        # Table headers
        f.write("""
        <tr>
        <th>Timestamp</th>
        <th>Host</th>
        <th>IP</th>
        <th>Port</th>
        <th>State</th>
        <th>Service</th>
        <th>Banner</th>
        <th>Notes</th>
        </tr>
        """)

        # Table rows
        for r in results:
            f.write("<tr>")
            f.write(f"<td>{r['timestamp']}</td>")
            f.write(f"<td>{r['host']}</td>")
            f.write(f"<td>{r['ip']}</td>")
            f.write(f"<td>{r['port']}</td>")
            f.write(f"<td>{r['state']}</td>")
            f.write(f"<td>{r['service']}</td>")
            f.write(f"<td>{r['banner']}</td>")
            f.write(f"<td>{r['notes']}</td>")
            f.write("</tr>")

        f.write("</table>")
        f.write("</body></html>")

    print(f"[+] HTML report saved: {filename}")

In [13]:
# -----------------------------------------------------------------------------------
# -----------------------------------------------------------------------------------
# SECTION 7 — Main Program
# -----------------------------------------------------------------------------------
# This function coordinates the entire scanning workflow. It handles:
# 1. Parsing user input
# 2. Resolving the target host
# 3. Running the multithreaded socket scan
# 4. Performing banner-based service detection
# 5. Optionally running an Nmap service detection scan
# 6. Aggregating scan results
# 7. Producing scan summary statistics
# 8. Exporting results to CSV and HTML reports
# -----------------------------------------------------------------------------------
# -----------------------------------------------------------------------------------
def main():
    start_time = datetime.now()                                         # Record scan start time so total scan duration can be calculated

    # --- Parse Command line arguments ---
    parser = argparse.ArgumentParser(description="A Complete Port Scanner")

    parser.add_argument("host", help="Target host to scan")                     # host - target hostname or IP
    parser.add_argument("ports", help="Port range to scan")                     # ports - port range
    parser.add_argument("--shodan", help="Optional Shodan API key")           # shodan → Shodan API key for OSINT enrichment(optional)


    # -----------------------------------------------------------------------------------
    # To allow program to run in both CLI and Jupyter Notebook
    # In notebook environments there are no CLI arguments available,
    # so default parameters are injected to allow the scanner to run.
    # -----------------------------------------------------------------------------------

    if "ipykernel" in sys.modules:
        print("[*] No CLI arguments detected. Entering Notebook mode and using default scan.")
        args = parser.parse_args(["scanme.nmap.org", "1-1000"])
    else:
        args = parser.parse_args()

    
    # Extract target host and parse the port range
    host = args.host
    start_port, end_port = map(int, args.ports.split("-"))

    ip = resolve_host(host)                                                      # Resolve hostname to IP address
    if not ip:
        return

    results = []                                                                 # This list will store all discovered scan results

    # --- Shodan lookup ---
    shodan_info = shodan_lookup(ip, args.shodan)


    # --- Multithreaded socket scanning with tqdm progress bar ---
    # Each port scan is executed in a separate thread.

    ports = range(start_port, end_port + 1)
    print(f"\n[+] Scanning {host} ({ip}) ports {start_port}-{end_port}...\n")

    executor = ThreadPoolExecutor(max_workers=250)                               # ThreadPoolExecutor manages a pool of worker threads
    futures = {executor.submit(scan_port, ip, port): port for port in ports}     # Submit a scan task for every port in the range

    iterator = as_completed(futures)                                             # Yields scan results as soon as each thread finishes

    if tqdm:                                                                     # tqdm provides a progress bar for long scans
        iterator = tqdm(iterator, total=len(futures), desc="Scanning")


     #-------- Process scan results as each thread completes ---------
    for future in iterator:                                                     

        port = futures[future]
        is_open, banner = future.result()                                         # Retrieve result from the completed scanning thread

        
        # Basic Service Identification
        # If a banner is received, simple keyword matching is used to infer common services.
        
        service_detected = ""
        if banner:
            banner_lower = banner.lower()
            if "ssh" in banner_lower:
                service_detected = "SSH"
            elif "http" in banner_lower:
                service_detected = "HTTP"
            elif "smtp" in banner_lower:
                service_detected = "SMTP"
            elif "ftp" in banner_lower:
                service_detected = "FTP"


        #------- If the port is open, record the scan result -----------

        if is_open:

            notes = "Socket scan"                                                   # Add metadata about the scan source
            if shodan_info["org"]:                                                  # Include Shodan organization information if available
                notes += f" | Org: {shodan_info['org']}"


        # Store result in structured dictionary format
            row = {
                "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                "host": host,
                "ip": ip,
                "port": port,
                "state": "open",
                "service": service_detected,
                "banner": banner,
                "notes": notes
            }

            results.append(row)

            #------------------------------------------------------------
            # Display open ports discovery with its service in real time
            #------------------------------------------------------------

            GREEN = "\033[92m"                                              # Using ANSI escape code to color the output green9readability)

            msg = f"{GREEN} ==>> OPEN PORT {port}"
            if service_detected:
                msg += f" ({service_detected})"
            if banner:
                msg += f" | Banner: {banner}"
            
            if tqdm:
                tqdm.write(msg)
            else:
                print(msg)


    #--------------------------------------------------------------------------------------------------
    # ----------# Ask the user if they want to perform deeper service detection using Nmap.------------
    # User response of y or n is required to proceed
    #---------------------------------------------------------------------------------------------------

    if tqdm:
        print()

    print("Do you want to run Nmap scan as well? (y/n): ")
    use_nmap = input()

    if use_nmap.lower() == "y":

        nmap_results = nmap_scan(host, f"{start_port}-{end_port}")                            # Run Nmap service detection

        # Add Nmap results to the overall scan dataset
        for r in nmap_results:

            print(f"{GREEN}[NMAP] Port {r['port']} {r['state']} | {r['service']} {r['version']}")

            row = {
                "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                "host": host,
                "ip": ip,
                "port": r["port"],
                "state": r["state"],
                "service": r["service"],
                "banner": r["banner"],
                "notes": "Detected by Nmap"
            }

            results.append(row)
            
    # ------------------------------------------------------------------
    # ------------------------------------------------------------------
    # Scan Summary
    #
    # The summary distinguishes between:
    # - ports discovered via socket scanning
    # - ports discovered via Nmap
    # - total unique open ports
    #
    # Sets are used to remove duplicates where both scanners detect
    # the same open port.
    # ----------------------------------------------------------------
    # ------------------------------------------------------------------

    total_open = len(results)
    
    # Count ports detected by each scanner
    # Socket ports
    socket_ports = sorted({r["port"] for r in results if "Socket" in r["notes"]})
    # Nmap ports
    nmap_ports   = sorted({r["port"] for r in results if "Nmap" in r["notes"]})


    # Count total unique open ports
    unique_count = len({r["port"] for r in results})

    print("\n---------------------------------")
    print("Scan Summary")
    print("---------------------------------")
    print(f"Target: {host} ({ip})")
    print(f"Ports scanned: {start_port}-{end_port}")
    print(f"Ports found by socket scanning: {len(socket_ports)} -> {socket_ports}")
    print(f"Ports found by Nmap: {len(nmap_ports)} -> {nmap_ports}")
    print(f"Total unique open ports: {unique_count}")
    print("---------------------------------")
    


    # --- Export results ---
    save_to_csv(results)
    save_to_html(results)

    # Calculate total scan duration
    end_time = datetime.now()
    duration = end_time - start_time

    print(f"Scan duration: {duration}")


# -----------------------------------------------------------------------------------
# Run program
# -----------------------------------------------------------------------------------
if __name__ == "__main__":
    main()

[*] No CLI arguments detected. Entering Notebook mode and using default scan.

[+] Scanning scanme.nmap.org (45.33.32.156) ports 1-1000...



Scanning:   0%|          | 1/1000 [00:00<04:07,  4.03it/s]

 ==>> OPEN PORT 22 (SSH) | Banner: SSH-2.0-OpenSSH_6.6.1p1 Ubuntu-2ubuntu2.13


Scanning:   0%|          | 2/1000 [00:00<05:19,  3.13it/s]

 ==>> OPEN PORT 80


Scanning: 100%|██████████| 1000/1000 [00:04<00:00, 247.37it/s]



Do you want to run Nmap scan as well? (y/n): 

[+] Running Nmap service detection scan (-sV)...
[+] Nmap scan completed.

[NMAP] Port 22 open | ssh 6.6.1p1 Ubuntu 2ubuntu2.13
[NMAP] Port 25 filtered | smtp 
[NMAP] Port 80 open | http 2.4.7

---------------------------------
Scan Summary
---------------------------------
Target: scanme.nmap.org (45.33.32.156)
Ports scanned: 1-1000
Ports found by socket scanning: 2 -> [22, 80]
Ports found by Nmap: 3 -> [22, 25, 80]
Total unique open ports: 3
---------------------------------

[+] CSV saved: scan_results.csv
[+] HTML report saved: scan_report.html
Scan duration: 0:00:23.471893


In [ ]:
####################################################################################################
# TESTING INSTRUCTIONS
#
# The following examples demonstrate how to run and test the scanner in both
# Jupyter Notebook and the Command Line Interface (CLI).
#
# Target used for testing:
# scanme.nmap.org
# (This is a safe public host provided by Nmap specifically for scanning practice.)
#
# -----------------------------------------------------------------------------------
# 1. TEST IN JUPYTER NOTEBOOK
# -----------------------------------------------------------------------------------
#
# Run the script cell. The program will automatically use default arguments:
#
#     Host  : scanme.nmap.org
#     Ports : 1-1000
#
# When prompted you can choose whether to run the Nmap scan.
#
# Example test cases:
#
# Test 1 — Socket scan only
#     Input: n
#
# Test 2 — Socket + Nmap scan
#     Input: y
#
# If you want to test Shodan in notebook mode, modify this line in main():
#
#     args = parser.parse_args(["scanme.nmap.org", "1-1000", "--shodan", "YOUR_API_KEY"])
#
#
# -----------------------------------------------------------------------------------
# 2. TEST FROM COMMAND LINE (CLI)
# -----------------------------------------------------------------------------------
#
# Navigate to the folder containing the scanner script and run:
#
# Socket scan only
#     python scanner.py scanme.nmap.org 1-1000
#
# Socket scan + Nmap
#     python scanner.py scanme.nmap.org 1-1000
#     When prompted type: y
#
# Socket scan + Shodan
#     python scanner.py scanme.nmap.org 1-1000 --shodan YOUR_API_KEY
#
# Full scan (Socket + Shodan + Nmap)
#     python scanner.py scanme.nmap.org 1-1000 --shodan YOUR_API_KEY
#     When prompted type: y
#
#
# -----------------------------------------------------------------------------------
# 3. EXPECTED OUTPUT
# -----------------------------------------------------------------------------------
#
# During scanning the program will display:
#
#  • Progress bar showing scan progress
#  • Open ports discovered
#  • Detected service banners
#  • Optional Nmap service/version detection
#
# After completion the scanner generates:
#
#  • CSV report   → scan_results.csv
#  • HTML report  → scan_report.html
#
# The scan summary displays:
#
#  • Target host and IP
#  • Port range scanned
#  • Ports detected via socket scanning
#  • Ports detected via Nmap
#  • Total unique open ports
#  • Total scan duration
#
####################################################################################################